In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("Connected to SQLite successfully.")

Connected to SQLite successfully.


In [4]:
# Reading the Superstore CSV file using pandas
# encoding="latin1" is used because this dataset has some special characters
# that don't load properly with the default utf-8 encoding
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

# checking shape - how many rows and columns we have
print(df.shape)

# preview first 5 rows of data
df.head()

(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
# Loading the raw dataframe into SQLite as a staging table
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

# confirming the table was created by counting rows
pd.read_sql("SELECT COUNT(*) AS total_rows FROM superstore_raw;", conn)

,total_rows
0,9994


In [6]:
# Creating the customers table using SELECT DISTINCT
# This avoids duplicate customer rows since one customer can have multiple orders
cursor.execute("DROP TABLE IF EXISTS customers;")
cursor.execute("""
CREATE TABLE customers AS
SELECT DISTINCT
    "Customer ID",
    "Customer Name",
    "Segment",
    "Country",
    "City",
    "State",
    "Postal Code",
    "Region"
FROM superstore_raw;
""")

pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)

,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South


In [7]:
# Creating the products table using SELECT DISTINCT
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("""
CREATE TABLE products AS
SELECT DISTINCT
    "Product ID",
    "Category",
    "Sub-Category",
    "Product Name"
FROM superstore_raw;
""")

pd.read_sql("SELECT * FROM products LIMIT 5;", conn)

,Product ID,Category,Sub-Category,Product Name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


In [8]:
# Creating the orders table - this acts as the main fact table
# Each row represents one order line item linked to a customer and a product
cursor.execute("DROP TABLE IF EXISTS orders;")
cursor.execute("""
CREATE TABLE orders AS
SELECT DISTINCT
    "Row ID",
    "Order ID",
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Customer ID",
    "Product ID",
    "Sales",
    "Quantity",
    "Discount",
    "Profit"
FROM superstore_raw;
""")

pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


In [9]:
# Quick check - counting rows in each table to confirm data loaded correctly
for t in ["customers", "orders", "products"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS rows FROM {t};", conn)
    print(t, "->", count.iloc[0,0])

customers -> 4910
orders -> 9994
products -> 1894


## Here Are The Required Queries

### Query 1: Find all orders where sales are greater than the average sales *(Subquery)*


In [10]:
# This uses a subquery - the inner query calculates average, outer query filters
query1 = """
SELECT *
FROM orders
WHERE Sales > (SELECT AVG(Sales) FROM orders);
"""
result1 = pd.read_sql(query1, conn)
print("Total orders above average sales:", len(result1))
result1.head()

Total orders above average sales: 2360


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
3,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
4,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092


### Query 2: Find the highest sales order for each customer *(Subquery)*

In [11]:
# Query 2: Find the highest sales order for each customer
# For each customer, this subquery checks the maximum sales value
# and the outer query returns the matching order row
query2 = """
SELECT o.*
FROM orders o
WHERE o.Sales = (
    SELECT MAX(o2.Sales)
    FROM orders o2
    WHERE o2."Customer ID" = o."Customer ID"
);
"""
result2 = pd.read_sql(query2, conn)
print("Total rows returned:", len(result2))
result2.head()

Total rows returned: 795


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
1,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
2,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
3,25,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,FUR-TA-10000577,1044.6300,3,0.00,240.2649
4,28,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,FUR-BO-10004834,3083.4300,7,0.50,-1665.0522


### Query 3: Calculate total sales for each customer *(CTE)*

In [12]:
# Query 3: Calculate total sales for each customer
# Using a CTE (Common Table Expression) - it lets us define a temporary
# named result set that we can then query like a normal table
query3 = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
)
SELECT * FROM customer_sales
ORDER BY Total_Sales DESC;
"""
result3 = pd.read_sql(query3, conn)
print("Total customers:", len(result3))
result3.head()

Total customers: 793


,Customer ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


### Query 4: Find customers whose total sales are above average *(CTE + Subquery)*

In [13]:
# Query 4: Find customers whose total sales are above average
# First we use a CTE to calculate each customer's total sales,
# then a subquery inside WHERE to compare against the overall average
query4 = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
)
SELECT *
FROM customer_sales
WHERE Total_Sales > (SELECT AVG(Total_Sales) FROM customer_sales)
ORDER BY Total_Sales DESC;
"""
result4 = pd.read_sql(query4, conn)
print("Customers with above-average total sales:", len(result4))
result4.head()

Customers with above-average total sales: 294


,Customer ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


### Query 5: Rank all customers based on total sales *(Window Function)*

In [14]:
#Query 5: Rank all customers based on total sales
#Using both RANK() and DENSE_RANK() to compare how they handle ties
query5 = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
)
SELECT 
    "Customer ID", 
    Total_Sales,
    RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank,
    DENSE_RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Dense_Rank
FROM customer_sales;
"""
result5 = pd.read_sql(query5, conn)
result5.head(10)

,Customer ID,Total_Sales,Sales_Rank,Sales_Dense_Rank
0,SM-20320,25043.050,1,1
1,TC-20980,19052.218,2,2
2,RB-19360,15117.339,3,3
3,TA-21385,14595.620,4,4
4,AB-10105,14473.571,5,5
5,KL-16645,14175.229,6,6
6,SC-20095,14142.334,7,7
7,HL-15040,12873.298,8,8
8,SE-20110,12209.438,9,9
9,CC-12370,12129.072,10,10


### Query 6: Assign row numbers to each order within a customer *(Window Function + PARTITION BY)*

In [15]:
# Query 6: Assign row numbers to each order within a customer
# PARTITION BY resets the numbering for each customer separately,
# so every customer's orders are numbered starting from 1
query6 = """
SELECT
    "Customer ID",
    "Order ID",
    "Order Date",
    Sales,
    ROW_NUMBER() OVER (
        PARTITION BY "Customer ID"
        ORDER BY "Order Date"
    ) AS Order_Sequence
FROM orders;
"""
result6 = pd.read_sql(query6, conn)
result6.head(10)

,Customer ID,Order ID,Order Date,Sales,Order_Sequence
0,AA-10315,CA-2015-121391,10/4/2015,26.960,1
1,AA-10315,CA-2016-103982,3/3/2016,3930.072,2
2,AA-10315,CA-2016-103982,3/3/2016,2.304,3
3,AA-10315,CA-2016-103982,3/3/2016,431.976,4
4,AA-10315,CA-2016-103982,3/3/2016,41.720,5
5,AA-10315,CA-2014-128055,3/31/2014,673.568,6
6,AA-10315,CA-2014-128055,3/31/2014,52.980,7
7,AA-10315,CA-2017-147039,6/29/2017,362.940,8
8,AA-10315,CA-2017-147039,6/29/2017,11.540,9
9,AA-10315,CA-2014-138100,9/15/2014,14.940,10


In [16]:
result6.sort_values(["Customer ID", "Order_Sequence"]).head(15)

,Customer ID,Order ID,Order Date,Sales,Order_Sequence
0,AA-10315,CA-2015-121391,10/4/2015,26.960,1
1,AA-10315,CA-2016-103982,3/3/2016,3930.072,2
2,AA-10315,CA-2016-103982,3/3/2016,2.304,3
3,AA-10315,CA-2016-103982,3/3/2016,431.976,4
4,AA-10315,CA-2016-103982,3/3/2016,41.720,5
5,AA-10315,CA-2014-128055,3/31/2014,673.568,6
6,AA-10315,CA-2014-128055,3/31/2014,52.980,7
7,AA-10315,CA-2017-147039,6/29/2017,362.940,8
8,AA-10315,CA-2017-147039,6/29/2017,11.540,9
9,AA-10315,CA-2014-138100,9/15/2014,14.940,10


### Query 7: Display top 3 customers based on total sales *(Window Function)*

In [17]:
# Query 7: Display top 3 customers based on total sales
# We rank customers first using a CTE, then filter for rank <= 3
# Note: window functions cannot be used directly in WHERE clause,
# so we wrap it in a CTE and filter in the outer query
query7 = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
),
ranked_customers AS (
    SELECT
        "Customer ID",
        Total_Sales,
        RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
    FROM customer_sales
)
SELECT * FROM ranked_customers
WHERE Sales_Rank <= 3;
"""
result7 = pd.read_sql(query7, conn)
result7

,Customer ID,Total_Sales,Sales_Rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


## Final Combined Query
Final query showing **Customer Name, Total Sales, Rank** — using JOIN + CTE + Window Function together.

In [18]:
# Final Combined Query: Shows Customer Name, Total Sales, and Rank
# Combines CTE (for aggregation), JOIN (to get customer name),
# and Window Function (for ranking) all together
final_query = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
)
SELECT
    c."Customer Name",
    cs.Total_Sales,
    RANK() OVER (ORDER BY cs.Total_Sales DESC) AS Rank
FROM customer_sales cs
JOIN customers c ON cs."Customer ID" = c."Customer ID"
ORDER BY Rank;
"""
final_result = pd.read_sql(final_query, conn)
final_result.head(10)

,Customer Name,Total_Sales,Rank
0,Sean Miller,25043.050,1
1,Sean Miller,25043.050,1
2,Sean Miller,25043.050,1
3,Sean Miller,25043.050,1
4,Sean Miller,25043.050,1
5,Tamara Chand,19052.218,6
6,Tamara Chand,19052.218,6
7,Tamara Chand,19052.218,6
8,Tamara Chand,19052.218,6
9,Tamara Chand,19052.218,6


## Mini Project: Customer Sales Insights

Answering business questions using SQL (Subqueries, CTEs, Window Functions).

**1. Who are the top 3 customers?**

In [19]:
# Reusing the final_query result - just taking the top 3 rows
top3 = final_result.head(3)
top3

,Customer Name,Total_Sales,Rank
0,Sean Miller,25043.05,1
1,Sean Miller,25043.05,1
2,Sean Miller,25043.05,1


**2. Who are the bottom 5 customers?**

In [20]:
# Query: Bottom 5 customers by total sales
bottom5_query = """
WITH customer_sales AS (
    SELECT "Customer ID", SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY "Customer ID"
)
SELECT c."Customer Name", cs.Total_Sales
FROM customer_sales cs
JOIN customers c ON cs."Customer ID" = c."Customer ID"
ORDER BY cs.Total_Sales ASC
LIMIT 5;
"""
bottom5 = pd.read_sql(bottom5_query, conn)
bottom5

,Customer Name,Total_Sales
0,Thais Sissman,4.833
1,Thais Sissman,4.833
2,Lela Donovan,5.304
3,Carl Jackson,16.520
4,Mitch Gastineau,16.739


**3. Which customers made only one order?**

In [21]:
# Query: Customers with only one order
# GROUP BY counts distinct Order IDs per customer,
# HAVING filters groups (not individual rows) where count = 1
single_order_query = """
SELECT c."Customer Name", COUNT(DISTINCT o."Order ID") AS Order_Count
FROM orders o
JOIN customers c ON o."Customer ID" = c."Customer ID"
GROUP BY c."Customer Name"
HAVING COUNT(DISTINCT o."Order ID") = 1;
"""
single_order_customers = pd.read_sql(single_order_query, conn)
print("Customers with only 1 order:", len(single_order_customers))
single_order_customers.head()

Customers with only 1 order: 12


,Customer Name,Order_Count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1


**4. Which customers have above-average sales?**

In [22]:
# Reusing Query 4's result from earlier - same logic applies here
above_avg_customers = result4
print("Count:", len(above_avg_customers))
above_avg_customers.head()

Count: 294


,Customer ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


**5. What is the highest order value per customer?**

In [23]:
# Query: Highest order value per customer
# Since one Order ID can have multiple line items (rows),
# we first sum sales per order, then take the max per customer
highest_order_value_query = """
WITH order_totals AS (
    SELECT "Customer ID", "Order ID", SUM(Sales) AS Order_Value
    FROM orders
    GROUP BY "Customer ID", "Order ID"
)
SELECT c."Customer Name", MAX(ot.Order_Value) AS Highest_Order_Value
FROM order_totals ot
JOIN customers c ON ot."Customer ID" = c."Customer ID"
GROUP BY c."Customer Name"
ORDER BY Highest_Order_Value DESC;
"""
highest_order_value = pd.read_sql(highest_order_value_query, conn)
highest_order_value.head(10)

,Customer Name,Highest_Order_Value
0,Sean Miller,23661.228
1,Tamara Chand,18336.740
2,Raymond Buch,14052.480
3,Tom Ashbrook,13716.458
4,Becky Martin,10539.896
5,Hunter Lopez,10499.970
6,Sanjit Chand,9900.190
7,Adrian Barton,9892.740
8,Bill Shonely,9135.190
9,Sanjit Engle,8805.040


## Key Insights

- A small group of top customers contribute a large share of total sales — clear evidence of an 80/20 pattern where few customers drive most revenue.
- A noticeable number of customers placed only a single order, which suggests an opportunity for retention or repeat-purchase campaigns.
- Customers with above-average total sales represent the core revenue base and could be prioritized for loyalty offers.
- The highest order value varies a lot across customers — some customers occasionally place large bulk orders rather than many small ones, which is useful to know for inventory planning.

In [24]:
# Closing the database connection - good practice once analysis is done
conn.close()
print("Connection closed.")

Connection closed.
